# Chapter 2: LLM API Basics

Companion notebook for Chapter 2. Each section corresponds to a listing in the book.

## Setup (Listing 2.1)

In [ ]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

%load_ext autoreload
%autoreload 2

## OpenAI API (Listing 2.2)

In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"}
    ]
)

print(response.choices[0].message.content)

## Anthropic API (Listing 2.3)

In [ ]:
from anthropic import Anthropic

client = Anthropic()

response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ]
)

print(response.content[0].text)

## Unified API with LiteLLM (Listing 2.4)

In [ ]:
from litellm import completion

response = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "Hello!"}]
)
print(response.choices[0].message.content)

response = completion(
    model="claude-sonnet-4-5",
    messages=[{"role": "user", "content": "Hello!"}]
)
print(response.choices[0].message.content)

## Demonstrating API statelessness (Listing 2.5)

In [ ]:
from litellm import completion

# First call
response1 = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "My name is Jungjun."}]
)
print(response1.choices[0].message.content)

# Second call
response2 = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "What is my name?"}]
)
print(response2.choices[0].message.content)

## Managing conversation history (Listing 2.6)

In [ ]:
from litellm import completion

messages = []

# First exchange
messages.append({"role": "user", "content": "My name is Jungjun."})
response1 = completion(model="gpt-5-mini", messages=messages)
assistant_message1 = response1.choices[0].message.content
messages.append({"role": "assistant", "content": assistant_message1})
print(assistant_message1)

# Second exchange - includes previous conversation history
messages.append({"role": "user", "content": "What is my name?"})
response2 = completion(model="gpt-5-mini", messages=messages)
assistant_message2 = response2.choices[0].message.content
print(assistant_message2)

## Structured output with Pydantic (Listing 2.7)

In [ ]:
from pydantic import BaseModel
from litellm import completion

class ExtractedInfo(BaseModel):
    name: str
    email: str
    phone: str | None = None

response = completion(
    model="gpt-5-mini",
    messages=[{
        "role": "user",
        "content": "My name is John Smith, my email is john@example.com, and my phone is 555-1234."
    }],
    response_format=ExtractedInfo
)

result = response.choices[0].message.content
print(result)

## Asynchronous LLM calls (Listing 2.8)

In [ ]:
import asyncio
from litellm import acompletion

async def get_response(prompt: str) -> str:
    response = await acompletion(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

prompts = [
    "What is 2 + 2?",
    "What is the capital of Japan?",
    "Who wrote Romeo and Juliet?"
]

# Execute all requests concurrently
tasks = [get_response(p) for p in prompts]
results = await asyncio.gather(*tasks)

for prompt, result in zip(prompts, results):
    print(f"Q: {prompt}")
    print(f"A: {result}\n")

## Rate-limited concurrent requests (Listing 2.9)

In [ ]:
import asyncio
from litellm import acompletion

# Limit to 10 concurrent requests
semaphore = asyncio.Semaphore(10)

async def call_llm(prompt: str) -> str:
    """LLM call with rate limiting and automatic retry."""
    async with semaphore:
        response = await acompletion(
            model="gpt-5-mini",
            messages=[{"role": "user", "content": prompt}],
            num_retries=3  # Automatic retry with exponential backoff
        )
        return response.choices[0].message.content

# Even with 100 concurrent tasks, only 10 API calls run at a time
prompts = [f"What is {i} + {i}?" for i in range(100)]
tasks = [call_llm(p) for p in prompts]
results = await asyncio.gather(*tasks, return_exceptions=True)

successes = sum(1 for r in results if not isinstance(r, Exception))
print(f"Successful: {successes}/{len(results)}")

## GAIA Benchmark Evaluation

Listings 2.10–2.17 are implemented in `scratch_agents.eval.gaia`. The notebook
imports the orchestration function and runs the experiment. Read the source
of `scratch_agents/eval/gaia.py` alongside the book listings for the full
implementation.

### Loading GAIA Level 1 problems (Listing 2.10)

In [ ]:
from datasets import load_dataset

level1_problems = load_dataset("gaia-benchmark/GAIA", "2023_level1", split="validation")
print(f"Number of Level 1 problems: {len(level1_problems)}")

### Model selection and execution (Listing 2.17)

In [ ]:
from scratch_agents.eval.gaia import run_experiment

MODELS = [
    "gpt-5",
    "gpt-5-mini",
    "anthropic/claude-sonnet-4-5",
    "anthropic/claude-haiku-4-5"
]

subset = level1_problems.select(range(20))
results = await run_experiment(subset, MODELS)

# Per-model accuracy
for model, model_results in results.items():
    correct = sum(1 for r in model_results if r["correct"])
    solvable = sum(1 for r in model_results if r.get("is_solvable"))
    total = len(model_results)
    print(f"{model}: {correct}/{total} correct, {solvable}/{total} judged solvable")